# Optuna Round 2 — worker `colab-gpu1` (Google Colab GPU)

Busca distribuída: **tcn_bi** rodam aqui; os wrappers leves ficam na GPU
local. Cada trial é gravado no sqlite (estudo `round2`), então desconexões do
Colab **não perdem progresso**.

## Setup (uma vez)
1. No computador local: `python scripts/colab_round2.py package` e faça upload
   do `dist/colab_round2_package_*.zip` para a pasta
   `MyDrive/wind-speed-colab/round2/package/` no drive.google.com.
2. **Runtime → Change runtime type → GPU** (sugestão: T4 (L4/A100 se disponivel — tcn_bi/transformer pesam)).
3. **Runtime → Run all**. O notebook restaura o progresso do Drive, roda o
   `step_optuna` com watchdog de RAM (reinicia o treino em caso de estouro,
   marcando trials órfãos como FAIL) e sincroniza sqlite + heartbeat de volta
   ao Drive a cada 10 min.

## Depois de uma desconexão
Rode tudo de novo — o progresso volta do Drive e o estudo continua.

O monitor local (`python scripts/colab_monitor.py --watch`) acompanha este
worker pelo heartbeat `heartbeats/colab-gpu1.json`. Mantenha esta aba aberta.


In [ ]:
# Parametros deste worker (gerado por scripts/colab_round2.py notebooks)
import json as _json

WORKER = _json.loads("""
{
  "worker": "colab-gpu1",
  "gpu_hint": "T4 (L4/A100 se disponivel \u2014 tcn_bi/transformer pesam)",
  "wrappers": [
    "tcn_bi"
  ],
  "config_path": "pipeline/pipeline.optuna_round2.json",
  "storage_subdir": "optuna_round2",
  "study_name": "round2",
  "n_trials": 120,
  "drive_root": "wind-speed-colab/round2",
  "wrapper_dirs": {
    "tcn_bi": "Seq2Seq_TCN_Bidirectional"
  },
  "poll_secs": 5,
  "heartbeat_secs": 60,
  "sync_secs": 600,
  "max_ram_gb": null
}
""")


In [ ]:
!nvidia-smi


In [ ]:
%pip install -q "tensorflow[and-cuda]==2.21.0" "keras==3.12.1" "keras-tcn==3.5.6"     "optuna==4.9.0" "PyWavelets==1.8.0" "pandas>=2.0" "scikit-learn>=1.3" "numpy>=1.26"
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))


In [ ]:
import shutil
import sqlite3
import zipfile
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

WORK = Path("/content")
REPO = WORK / "wind-speed-forecasting"
DRIVE = Path("/content/drive/MyDrive") / WORKER["drive_root"]
PKG_DIR = DRIVE / "package"
PKG_DIR.mkdir(parents=True, exist_ok=True)

zips = sorted(PKG_DIR.glob("colab_round2_package_*.zip"), key=lambda p: p.stat().st_mtime)
assert zips, (
    f"Nenhum colab_round2_package_*.zip em {PKG_DIR}.\n"
    "Rode 'python scripts/colab_round2.py package' localmente e suba o zip para la."
)
pkg = zips[-1]
print("pacote:", pkg.name)
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(pkg) as z:
    z.extractall(REPO)
assert (REPO / "data" / "dataset.csv").is_file(), "dataset.csv ausente no pacote"

# Restaura o progresso ja sincronizado para o Drive. O backup API do sqlite
# garante uma copia consistente mesmo se o arquivo de origem for um snapshot
# pego no meio de uma escrita.
DEST = REPO / "pipeline" / "tmp" / WORKER["storage_subdir"]
optuna_drive = DRIVE / "optuna"
if optuna_drive.is_dir():
    for src_dir in sorted(p for p in optuna_drive.iterdir() if p.is_dir()):
        db = src_dir / "optuna.db"
        if not db.is_file():
            continue
        dst = DEST / src_dir.name / "optuna.db"
        dst.parent.mkdir(parents=True, exist_ok=True)
        src_conn = sqlite3.connect(db)
        dst_conn = sqlite3.connect(dst)
        with dst_conn:
            src_conn.backup(dst_conn)
        src_conn.close()
        dst_conn.close()
        for f in src_dir.glob("*.json"):
            shutil.copy2(f, dst.parent / f.name)
        print("progresso restaurado:", src_dir.name)
print("repo em:", REPO)


In [ ]:
# Watchdog do worker: roda o step_optuna como subprocesso, reinicia em estouro
# de RAM/crash (o estudo sempre resume do sqlite) e publica heartbeat + copia
# do progresso no Drive.
import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

os.chdir(REPO)
os.environ.setdefault("MALLOC_TRIM_THRESHOLD_", "134217728")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

LOG = Path("pipeline/tmp/colab_round2/logs") / (WORKER["worker"] + ".log")
LOG.parent.mkdir(parents=True, exist_ok=True)
HB_PATH = DRIVE / "heartbeats" / (WORKER["worker"] + ".json")
HB_PATH.parent.mkdir(parents=True, exist_ok=True)
STORAGE = Path("pipeline") / "tmp" / WORKER["storage_subdir"]
POLL = int(WORKER.get("poll_secs", 5))
HB_EVERY = int(WORKER.get("heartbeat_secs", 60))
SYNC_EVERY = int(WORKER.get("sync_secs", 600))
WRAPPER_DIRS_LOCAL = WORKER["wrapper_dirs"]


def _meminfo():
    info = {}
    with open("/proc/meminfo") as handle:
        for line in handle:
            key, value = line.split(":", 1)
            info[key] = int(value.strip().split()[0])  # kB
    return info


def total_ram_gb():
    return _meminfo()["MemTotal"] / 1024 / 1024


def used_ram_gb():
    info = _meminfo()
    return (info["MemTotal"] - info["MemAvailable"]) / 1024 / 1024


def gpu_stats():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip().split(",")
        return {"name": out[0].strip(), "mem_used_mb": int(out[1]),
                "mem_total_mb": int(out[2]), "util_pct": int(out[3])}
    except Exception:
        return None


def tail_lines(path, n=25):
    try:
        with open(path, "rb") as handle:
            handle.seek(0, os.SEEK_END)
            end = handle.tell()
            window = min(end, 64 * 1024)
            handle.seek(end - window)
            return handle.read().decode("utf-8", "replace").splitlines()[-n:]
    except OSError:
        return []


ERROR_RE = re.compile(
    "ResourceExhaustedError|OutOfMemoryError|OOM was thrown|CUDA_ERROR|"
    "Xid|InternalError|ImportError|ModuleNotFoundError|FileNotFoundError|"
    "Traceback \(most recent call last\)|Killed|sqlite3\.|OperationalError",
    re.IGNORECASE,
)
OPTIMIZING_RE = re.compile(r"Optimizing (\w+)")

HB = {
    "worker": WORKER["worker"],
    "phase": "starting",
    "updated_at": None,
    "restarts": 0,
    "current_wrapper": None,
    "wrappers": {},
    "gpu": None,
    "ram_gb": {},
    "recent_errors": [],
    "last_log_lines": [],
    "package": Path(pkg).name,
}


def write_hb(**updates):
    HB.update(updates)
    HB["updated_at"] = datetime.now(timezone.utc).isoformat()
    HB["ram_gb"] = {"used": round(used_ram_gb(), 1), "total": round(total_ram_gb(), 1)}
    HB["gpu"] = gpu_stats()
    HB["last_log_lines"] = tail_lines(LOG)
    hits = [line for line in HB["last_log_lines"] if ERROR_RE.search(line)]
    if hits:
        stamp = HB["updated_at"]
        for line in hits[-3:]:
            HB["recent_errors"] = (
                [{"at": stamp, "line": line[:200]}] + HB["recent_errors"]
            )[:10]
    tmp = HB_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(HB, indent=1), encoding="utf-8")
    tmp.replace(HB_PATH)


def sqlite_backup(src, dst):
    import sqlite3
    dst.parent.mkdir(parents=True, exist_ok=True)
    src_conn = sqlite3.connect(src)
    dst_conn = sqlite3.connect(dst)
    with dst_conn:
        src_conn.backup(dst_conn)
    src_conn.close()
    dst_conn.close()


def sync_progress():
    if not STORAGE.is_dir():
        return
    for db in STORAGE.glob("*/optuna.db"):
        try:
            sqlite_backup(db, DRIVE / "optuna" / db.parent.name / "optuna.db")
            for f in db.parent.glob("*.json"):
                shutil.copy2(f, DRIVE / "optuna" / db.parent.name / f.name)
        except Exception as exc:  # Drive pode cair em sessao longa; nao derruba o treino
            HB["recent_errors"] = (
                [{"at": HB["updated_at"], "line": "sync_drive: " + str(exc)[:200]}]
                + HB["recent_errors"]
            )[:10]


def budget_state():
    import sqlite3
    state = {}
    for wrapper in WORKER["wrappers"]:
        wrapper_dir = WRAPPER_DIRS_LOCAL[wrapper]
        db = STORAGE / wrapper_dir / "optuna.db"
        trials = None
        if db.is_file():
            try:
                conn = sqlite3.connect(db)
                rows = conn.execute(
                    "SELECT COUNT(*) FROM trials t "
                    "JOIN studies s ON t.study_id = s.study_id WHERE s.study_name = ?",
                    (WORKER["study_name"],),
                ).fetchone()
                trials = rows[0]
                conn.close()
            except sqlite3.Error:
                trials = None
        state[wrapper] = {"trials": trials, "budget": WORKER["n_trials"],
                          "done": trials is not None and trials >= WORKER["n_trials"]}
    return state


def fail_orphan_trials():
    import sqlite3
    for db in STORAGE.glob("*/optuna.db"):
        try:
            conn = sqlite3.connect(db)
            conn.execute(
                "UPDATE trials SET state = 'FAIL' "
                "WHERE state = 'RUNNING' AND study_id IN "
                "(SELECT study_id FROM studies WHERE study_name = ?)",
                (WORKER["study_name"],),
            )
            conn.commit()
            conn.close()
        except sqlite3.Error as exc:
            print("fail_orphans", db, "->", exc)


MAX_RAM_GB = WORKER.get("max_ram_gb") or round(total_ram_gb() * 0.82, 1)
print(f"worker={WORKER['worker']} wrappers={WORKER['wrappers']} "
      f"budget={WORKER['n_trials']} trials | limite RAM {MAX_RAM_GB} GB "
      f"(total {total_ram_gb():.1f} GB)")

fail_orphan_trials()
sync_progress()
restarts = 0
fast_crashes = 0
backoff = 5
while True:
    write_hb(phase="running", restarts=restarts,
             wrappers=budget_state(), current_wrapper=None)
    if all(info["done"] for info in HB["wrappers"].values()):
        print("orcamento alcancado para todos os wrappers deste worker.")
        break

    cmd = [sys.executable, "pipeline/step_optuna.py", "--config", WORKER["config_path"],
           "--wrappers", *WORKER["wrappers"]]
    print("\n$ " + " ".join(cmd))
    log_handle = open(LOG, "ab")
    started = time.time()
    proc = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT,
                            env=dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2"))
    killed = False
    last_hb = last_sync = 0.0
    while proc.poll() is None:
        time.sleep(POLL)
        now = time.time()
        if now - last_hb >= HB_EVERY:
            last_hb = now
            log_tail = tail_lines(LOG, 40)
            current = None
            for line in log_tail:
                found = OPTIMIZING_RE.search(line)
                if found:
                    current = found.group(1)
            write_hb(phase="running", restarts=restarts,
                     wrappers=budget_state(), current_wrapper=current)
            counts = ", ".join("{}:{}".format(k, v["trials"])
                               for k, v in HB["wrappers"].items())
            gpu_mb = HB["gpu"]["mem_used_mb"] if HB["gpu"] else "?"
            print("[hb] {} ram={}GB gpu={}MB wrapper={} trials=({})".format(
                datetime.now().strftime("%H:%M:%S"), HB["ram_gb"]["used"],
                gpu_mb, HB["current_wrapper"], counts), flush=True)
        if now - last_sync >= SYNC_EVERY:
            last_sync = now
            sync_progress()
        if used_ram_gb() > MAX_RAM_GB:
            print(f"RAM {used_ram_gb():.1f}GB > {MAX_RAM_GB}GB -> kill + restart")
            proc.terminate()
            time.sleep(3)
            proc.kill()
            killed = True
            break
    rc = proc.wait()
    log_handle.close()
    runtime = time.time() - started
    write_hb(phase="restarting" if rc != 0 else "finishing", restarts=restarts,
             wrappers=budget_state())

    if rc == 0 and not killed:
        # rc 0 = passo completo do step_optuna (orcamento OU timeout_sec): o
        # worker termina; um re-run do notebook retoma o que faltar.
        state = budget_state()
        write_hb(phase="done", wrappers=state)
        print("step_optuna terminou (rc=0).")
        break

    restarts += 1
    fail_orphan_trials()
    sync_progress()
    if runtime < 120:
        fast_crashes += 1
    else:
        fast_crashes = 0
        backoff = 5
    if fast_crashes >= 5:
        write_hb(phase="error",
                 last_error="5 reinicios rapidos consecutivos; veja " + str(LOG))
        raise RuntimeError("watchdog: crash loop detectado (veja o log do worker)")
    print(f"subprocesso rc={rc} killed={killed} runtime={runtime:.0f}s -> "
          f"restart #{restarts} em {backoff}s")
    time.sleep(backoff)
    backoff = min(backoff * 2, 60)

sync_progress()
state = budget_state()
write_hb(phase="done", wrappers=state)
print("\nResumo final:")
for wrapper, info in state.items():
    print(f"  {wrapper}: {info['trials']}/{info['budget']} trials")


In [ ]:
import shutil
from pathlib import Path

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
STORAGE = Path("pipeline/tmp") / WORKER["storage_subdir"]
for db in sorted(STORAGE.glob("*/optuna.db")):
    try:
        study = optuna.load_study(study_name=WORKER["study_name"], storage=f"sqlite:///{db}")
        done = [t for t in study.trials if t.state.name == "COMPLETE"]
        best = min((t.value for t in done), default=float("nan"))
        print(f"{db.parent.name}: {len(study.trials)} trials | melhor={best:.5f}")
    except Exception as exc:
        print(f"{db.parent.name}: {exc}")

# zip final tambem no Drive (copia completa, independente do sync periodico)
archive = Path("/content") / f"{WORKER['storage_subdir']}_{WORKER['worker']}"
shutil.make_archive(str(archive), "zip", ".", str(STORAGE))
shutil.copy2(str(archive) + ".zip", DRIVE / (archive.name + ".zip"))
print("zip salvo no Drive:", archive.name + ".zip")
try:
    from google.colab import files
    files.download(str(archive) + ".zip")
except Exception as exc:
    print("download automatico falhou (pegue no Drive):", exc)
